In [ ]:
import pandas as pd
import numpy as np
import re
import os

RAW       = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\raw"
PROCESSED = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\processed"

# ── helpers ──────────────────────────────────────────────
def standardize_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

def clean_city_name(name):
    if pd.isna(name): return name
    name = str(name).strip().lower()
    for a, b in {'ã':'a','á':'a','à':'a','â':'a','é':'e','è':'e','ê':'e',
                 'í':'i','ì':'i','ó':'o','ô':'o','õ':'o','ú':'u','ü':'u',
                 'ç':'c','ñ':'n'}.items():
        name = name.replace(a, b)
    return re.sub(r'\s+', ' ', name).strip()

global_overrides = {
    'new york city':'new york','bombay':'mumbai',
    'calcutta':'kolkata','bangalore':'bengaluru','peking':'beijing',
}

def apply_city_clean(df, col='city'):
    df[col] = df[col].apply(clean_city_name).replace(global_overrides)
    return df

# ── load ─────────────────────────────────────────────────
aqi   = standardize_columns(pd.read_csv(os.path.join(RAW, "global air pollution dataset.csv")))
col   = standardize_columns(pd.read_csv(os.path.join(RAW, "cost-of-living_v2.csv")))
pop1  = standardize_columns(pd.read_csv(os.path.join(RAW, "CCCdata1.csv")))
pop2  = standardize_columns(pd.read_csv(os.path.join(RAW, "CCCdata2.csv")))
crime = standardize_columns(pd.read_csv(os.path.join(RAW, "crime-index-2023.csv")))

# ── clean city names ──────────────────────────────────────
for d in [aqi, col, pop1, pop2, crime]:
    apply_city_clean(d)

# ── pop1 overrides BEFORE merge ───────────────────────────
pop1_overrides = {
    'birmingham-wolverhampton':'birmingham',
    'dallas-fort worth':'dallas',
    'leeds-bradford':'leeds',
    'cape coral-fort myers':'fort myers',
    'greensboro-high point':'greensboro',
    'newcastle-sunderland':'newcastle upon tyne',
    'omaha-council bluffs':'omaha',
    'oxnard-thousand oaks-ventura':'oxnard',
    'gdansk, gdynia and sopot':'gdansk',
    'sendai, miyagi':'sendai',
    'ruhr region east':'dortmund',
    'ruhr region west':'essen',
    'katowice urban area':'katowice',
    'moscow region (oblast)':'moscow',
    'brighton and hove':'brighton',
    'frankfurt am main':'frankfurt',
    'saint petersburg':'st. petersburg',
    'saint-etienne':'saint etienne',
    'palma de mallorca':'palma',
    'vitoria-gasteiz':'vitoria',
    'nur-sultan':'astana',
    'new delhi':'delhi',
    'new taipei city':'taipei',
    'xian':"xi'an",
    'macau':'macao',
    'penang':'george town',
    'kuwait city':'kuwait',
    'luxembourg':'luxembourg city',
    'kyiv':'kiev',
    'odessa':'odesa',
    'kharkiv':'kharkov',
    'the hague':'den haag',
    'quezon city':'quezon',
}
pop1['city'] = pop1['city'].replace(pop1_overrides)

# ── rename cost of living columns ─────────────────────────
col_rename = {
    'x1':  'meal_cheap_usd',
    'x2':  'meal_midrange_usd',
    'x3':  'mcmeal_usd',
    'x4':  'beer_local_usd',
    'x5':  'beer_import_usd',
    'x6':  'cappuccino_usd',
    'x7':  'coke_usd',
    'x8':  'water_usd',
    'x9':  'milk_1l_usd',
    'x10': 'bread_usd',
    'x11': 'rice_1kg_usd',
    'x12': 'eggs_12_usd',
    'x13': 'cheese_1kg_usd',
    'x14': 'chicken_1kg_usd',
    'x15': 'beef_1kg_usd',
    'x16': 'apples_1kg_usd',
    'x17': 'banana_1kg_usd',
    'x18': 'oranges_1kg_usd',
    'x19': 'tomato_1kg_usd',
    'x20': 'potato_1kg_usd',
    'x21': 'onion_1kg_usd',
    'x22': 'lettuce_usd',
    'x23': 'water_bottle_usd',
    'x24': 'wine_bottle_usd',
    'x25': 'domestic_beer_usd',
    'x26': 'cigarettes_usd',
    'x27': 'cigarettes_20_usd',
    'x28': 'oneway_ticket_usd',
    'x29': 'monthly_pass_usd',        
    'x30': 'taxi_start_usd',
    'x31': 'taxi_1km_usd',
    'x32': 'taxi_1hr_wait_usd',
    'x33': 'petrol_1l_usd',
    'x34': 'vw_golf_usd',
    'x35': 'toyota_corolla_usd',      
    'x36': 'basic_utilities_usd',     
    'x37': 'mobile_min_usd',
    'x38': 'internet_usd',
    'x39': 'fitness_club_usd',
    'x40': 'tennis_court_usd',
    'x41': 'cinema_usd',
    'x42': 'preschool_usd',
    'x43': 'intl_school_usd',
    'x44': 'jeans_usd',               
    'x45': 'summer_dress_usd',
    'x46': 'nike_shoes_usd',
    'x47': 'leather_shoes_usd',
    'x48': 'rent_1br_city_usd',       
    'x49': 'rent_1br_outside_usd',    
    'x50': 'rent_3br_city_usd',
    'x51': 'rent_3br_outside_usd',
    'x52': 'price_sqm_city_usd',
    'x53': 'price_sqm_outside_usd',
    'x54': 'avg_net_salary_usd',      
    'x55': 'mortgage_rate_pct',
}
col = col.rename(columns=col_rename)

# ── select columns ────────────────────────────────────────
aqi_c = aqi[['country','city','aqi_value','aqi_category','pm2.5_aqi_value']]\
            .dropna(subset=['city']).drop_duplicates(subset=['city','country'])

col_c = col[['city','country','meal_cheap_usd','meal_midrange_usd',
             'rent_1br_outside_usd','rent_3br_city_usd','basic_utilities_usd',
             'monthly_pass_usd','avg_net_salary_usd',
             'data_quality']].copy()
col_c = col_c[col_c['data_quality']==1].drop_duplicates(subset=['city','country'])

pop1_c = pop1[['city','country','popcity','densitycity','congestion2019']]\
              .drop_duplicates(subset=['city','country'])

crime_c = crime[['city','country','crime_index','safety_index']]\
                .drop_duplicates(subset=['city','country'])

# ── check overlap before merge ────────────────────────────
merged_cities = set(aqi_c.merge(col_c.drop(columns='country'), on='city', how='inner')['city'])
pop1_cities   = set(pop1_c['city'])
print(f"Cities going into merge: {len(merged_cities)}")
print(f"Pop1 cities available:   {len(pop1_cities)}")
print(f"Overlap:                 {len(merged_cities & pop1_cities)}")

# ── merge ─────────────────────────────────────────────────
df = aqi_c.copy()
df = df.merge(col_c.drop(columns='country'),   on='city', how='inner')
df = df.merge(pop1_c.drop(columns='country'),  on='city', how='left')
df = df.merge(crime_c.drop(columns='country'), on='city', how='left')

# ── impute & clean ────────────────────────────────────────
df['crime_index']         = df['crime_index'].fillna(df['crime_index'].median())
df['safety_index']        = df['safety_index'].fillna(df['safety_index'].median())
df['congestion2019']      = df['congestion2019'].fillna(df['congestion2019'].median())
df['country']             = df['country'].fillna('Unknown')
df = df.drop(columns=['data_quality'], errors='ignore')
df = df.dropna(subset=['avg_net_salary_usd','basic_utilities_usd','monthly_pass_usd'])

# ── drop cities without real density ─────────────────────
df_final = df.dropna(subset=['popcity','densitycity'])

print(f"\nFinal city count: {df_final.shape[0]}")
print(f"Columns:          {df_final.shape[1]}")
print(f"Any nulls left:   {df_final.isnull().sum().sum()}")
print(f"\nSample:\n{df_final[['city','country','aqi_value','densitycity','crime_index','avg_net_salary_usd']].head(10)}")

Cities going into merge: 796
Pop1 cities available:   508
Overlap:                 338


KeyError: 'quality_of_life_index'

In [43]:
import pandas as pd
import numpy as np
import re
import os

RAW       = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\raw"
PROCESSED = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\processed"

# ── helpers ──────────────────────────────────────────────
def standardize_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

def clean_city_name(name):
    if pd.isna(name): return name
    name = str(name).strip().lower()
    for a, b in {'ã':'a','á':'a','à':'a','â':'a','é':'e','è':'e','ê':'e',
                 'í':'i','ì':'i','ó':'o','ô':'o','õ':'o','ú':'u','ü':'u',
                 'ç':'c','ñ':'n'}.items():
        name = name.replace(a, b)
    return re.sub(r'\s+', ' ', name).strip()

global_overrides = {
    'new york city':'new york','bombay':'mumbai',
    'calcutta':'kolkata','bangalore':'bengaluru','peking':'beijing',
}

def apply_city_clean(df, col='city'):
    df[col] = df[col].apply(clean_city_name).replace(global_overrides)
    return df

# ── load ─────────────────────────────────────────────────
aqi   = standardize_columns(pd.read_csv(os.path.join(RAW, "global air pollution dataset.csv")))
col   = standardize_columns(pd.read_csv(os.path.join(RAW, "cost-of-living_v2.csv")))
pop1  = standardize_columns(pd.read_csv(os.path.join(RAW, "CCCdata1.csv")))
pop2  = standardize_columns(pd.read_csv(os.path.join(RAW, "CCCdata2.csv")))
crime = standardize_columns(pd.read_csv(os.path.join(RAW, "crime-index-2023.csv")))

# ── clean city names ──────────────────────────────────────
for d in [aqi, col, pop1, pop2, crime]:
    apply_city_clean(d)

# ── pop1 overrides BEFORE merge ───────────────────────────
pop1_overrides = {
    'birmingham-wolverhampton':'birmingham',
    'dallas-fort worth':'dallas',
    'leeds-bradford':'leeds',
    'cape coral-fort myers':'fort myers',
    'greensboro-high point':'greensboro',
    'newcastle-sunderland':'newcastle upon tyne',
    'omaha-council bluffs':'omaha',
    'oxnard-thousand oaks-ventura':'oxnard',
    'gdansk, gdynia and sopot':'gdansk',
    'sendai, miyagi':'sendai',
    'ruhr region east':'dortmund',
    'ruhr region west':'essen',
    'katowice urban area':'katowice',
    'moscow region (oblast)':'moscow',
    'brighton and hove':'brighton',
    'frankfurt am main':'frankfurt',
    'saint petersburg':'st. petersburg',
    'saint-etienne':'saint etienne',
    'palma de mallorca':'palma',
    'vitoria-gasteiz':'vitoria',
    'nur-sultan':'astana',
    'new delhi':'delhi',
    'new taipei city':'taipei',
    'xian':"xi'an",
    'macau':'macao',
    'penang':'george town',
    'kuwait city':'kuwait',
    'luxembourg':'luxembourg city',
    'kyiv':'kiev',
    'odessa':'odesa',
    'kharkiv':'kharkov',
    'the hague':'den haag',
    'quezon city':'quezon',
}
pop1['city'] = pop1['city'].replace(pop1_overrides)

# ── rename cost of living columns ─────────────────────────
col_rename = {
    'x1':'meal_cheap_usd','x2':'meal_midrange_usd','x3':'mcmeal_usd',
    'x4':'beer_local_usd','x5':'beer_import_usd','x6':'cappuccino_usd',
    'x7':'coke_usd','x8':'water_usd','x9':'milk_1l_usd','x10':'bread_usd',
    'x11':'rice_1kg_usd','x12':'eggs_12_usd','x13':'cheese_1kg_usd',
    'x14':'chicken_1kg_usd','x15':'beef_1kg_usd','x16':'apples_1kg_usd',
    'x17':'banana_1kg_usd','x18':'oranges_1kg_usd','x19':'tomato_1kg_usd',
    'x20':'potato_1kg_usd','x21':'onion_1kg_usd','x22':'lettuce_usd',
    'x23':'water_bottle_usd','x24':'wine_bottle_usd','x25':'domestic_beer_usd',
    'x26':'cigarettes_usd','x27':'oneway_ticket_usd','x28':'monthly_pass_usd',
    'x29':'taxi_start_usd','x30':'taxi_1km_usd','x31':'taxi_1hr_wait_usd',
    'x32':'petrol_1l_usd','x33':'volkswagen_golf_usd','x34':'toyota_corolla_usd',
    'x35':'basic_utilities_usd','x36':'mobile_plan_usd','x37':'internet_usd',
    'x38':'fitness_club_usd','x39':'tennis_court_usd','x40':'cinema_usd',
    'x41':'preschool_usd','x42':'intl_school_usd',
    'x43':'rent_1br_city_local',
    'x44':'rent_1br_outside_usd','x45':'rent_3br_city_usd',
    'x46':'rent_3br_outside_usd','x47':'price_sqm_city_usd',
    'x48':'price_sqm_outside_usd','x49':'mortgage_rate_pct',
    'x50':'avg_net_salary_usd',
    'x51':'purchasing_power_local','x52':'purchasing_power_incl_rent',
    'x53':'purchasing_power_index','x54':'rent_index',
    'x55':'quality_of_life_index',
}
col = col.rename(columns=col_rename)

# ── select columns ────────────────────────────────────────
aqi_c = aqi[['country','city','aqi_value','aqi_category','pm2.5_aqi_value']]\
            .dropna(subset=['city']).drop_duplicates(subset=['city','country'])

col_c = col[['city','country','meal_cheap_usd','meal_midrange_usd',
             'rent_1br_outside_usd','rent_3br_city_usd','basic_utilities_usd',
             'monthly_pass_usd','avg_net_salary_usd','quality_of_life_index',
             'data_quality']].copy()
col_c = col_c[col_c['data_quality']==1].drop_duplicates(subset=['city','country'])

pop1_c = pop1[['city','country','popcity','densitycity','congestion2019']]\
              .drop_duplicates(subset=['city','country'])

crime_c = crime[['city','country','crime_index','safety_index']]\
                .drop_duplicates(subset=['city','country'])

# ── check overlap before merge ────────────────────────────
merged_cities = set(aqi_c.merge(col_c.drop(columns='country'), on='city', how='inner')['city'])
pop1_cities   = set(pop1_c['city'])
print(f"Cities going into merge: {len(merged_cities)}")
print(f"Pop1 cities available:   {len(pop1_cities)}")
print(f"Overlap:                 {len(merged_cities & pop1_cities)}")

# ── merge ─────────────────────────────────────────────────
df = aqi_c.copy()
df = df.merge(col_c.drop(columns='country'),   on='city', how='inner')
df = df.merge(pop1_c.drop(columns='country'),  on='city', how='left')
df = df.merge(crime_c.drop(columns='country'), on='city', how='left')

# ── impute & clean ────────────────────────────────────────
df['crime_index']         = df['crime_index'].fillna(df['crime_index'].median())
df['safety_index']        = df['safety_index'].fillna(df['safety_index'].median())
df['congestion2019']      = df['congestion2019'].fillna(df['congestion2019'].median())
df['quality_of_life_index'] = df['quality_of_life_index'].fillna(df['quality_of_life_index'].median())
df['country']             = df['country'].fillna('Unknown')
df = df.drop(columns=['data_quality'], errors='ignore')
df = df.dropna(subset=['avg_net_salary_usd','basic_utilities_usd','monthly_pass_usd'])

# ── drop cities without real density ─────────────────────
df_final = df.dropna(subset=['popcity','densitycity'])

print(f"\nFinal city count: {df_final.shape[0]}")
print(f"Columns:          {df_final.shape[1]}")
print(f"Any nulls left:   {df_final.isnull().sum().sum()}")
print(f"\nSample:\n{df_final[['city','country','aqi_value','densitycity','crime_index','avg_net_salary_usd']].head(10)}")

Cities going into merge: 796
Pop1 cities available:   508
Overlap:                 338

Final city count: 356
Columns:          18
Any nulls left:   0

Sample:
             city                      country  aqi_value  densitycity  \
0         phoenix     United States of America         72  1077.541931   
2   dar es salaam  United Republic of Tanzania         51  3133.195262   
4        hangzhou                        China        203  1013.691165   
6         tianjin                        China        142  1325.031839   
10         fuzhou                        China        168  2231.484044   
17       salvador                       Brazil         30  3786.935484   
20       richmond     United States of America         76  1315.768898   
21       richmond     United States of America         76  1315.768898   
23         lublin                       Poland         32  2421.905731   
24        hamburg                      Germany         34  2366.487488   

    crime_index  avg_net_

In [44]:
output_path = os.path.join(PROCESSED, "daedalus_merged.csv")
df_final.to_csv(output_path, index=False)
print(f"Saved → {output_path}")

Saved → C:\Users\alokhande\Desktop\Project_Daedalus\data\processed\daedalus_merged.csv
